In [1]:
from chat_utils import *

In [45]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format: "python" or "json" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    result = chat_message(messages, stop_sequences=["```"])

    return json.loads(result)

In [46]:
generated_dataset = generate_dataset()
with open("dataset.json", 'w') as file:
    json.dump(generated_dataset, file, indent=2)

In [33]:
def run_prompt(test_case):
    """Merges the prompt with the test case and returns the result"""
    prompt = f"""
Please solve the following task:

{test_case['task']}

* Respond only with Python, JSON, or a regular expression
* Do not add any comments or commentary on explanation
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, f"```{test_case['format']}")
    return chat_message(messages, stop_sequences=["```"])

In [41]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    code_score = grade_syntax(test_case, output)

    model_grade = grade_by_model(test_case, output)

    print(model_grade)

    model_score = model_grade['score']
    model_reasoning = model_grade['reasoning']

    average_score = (model_score + code_score) / 2
    return {
        "test_case": test_case,
        "score": average_score,
        "output": output,
        "reasoning": model_reasoning,
    }

In [13]:
def run_eval(dataset):
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [48]:
with open("dataset.json", 'r') as file:
    eval_dataset = json.load(file)

eval_result = run_eval(dataset=eval_dataset)

with open("eval_result.json", 'w') as file:
    json.dump(eval_result, file, indent=2)


{
    "strengths": [
        "Correctly enforces the required prefix 'aws-logs-v2/' using an anchor.",
        "Properly validates the mandatory '/0/' suffix at the end of the specific pattern segment.",
        "Allows for standard alphanumeric characters and hyphens in the identifier, covering common naming conventions."
    ],
    "weaknesses": [
        "Contains a trailing '.*' which violates the constraint to match *exactly* one identifier segment by potentially matching additional arbitrary text after '/0/'.",
        "Fails to strictly enforce that the pattern ends immediately after '/0/', risking false positives on logs with extra data appended."
    ],
    "reasoning": "The regex is mostly accurate but fails the 'exact match' constraint due to the trailing wildcard '.*'. The solution allows matching patterns like 'aws-logs-v2/my-id/0/something', whereas the task likely implies the stream name structure ends at '/0/'. Removing the trailing '.*' would strictly enforce the boun

In [47]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<solution_criteria>
{test_case["solution_criteria"]}
</solution_criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat_message(messages, stop_sequences=["```"])
    print(eval_text)
    return json.loads(eval_text)

In [28]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(test_case, response):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)
